<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

# 📑 Table of Contents: E-002 Full ICD-10 Classification

---

### 🧠 Bio_ClinicalBERT for Full ICD-10 Classification
Experiment objective, strategy, and comparison with E-001.

### 🧪 Experiment Log: Scientific Record (E-002)
Baseline configuration and official results.

### 🔬 Phase 1: Experiment Configuration
MLflow SQLite backend, Gold layer Parquet discovery, hyperparameters,
device selection. Filters to billable records only (`code_status == "billable"`).

### ⚙️ Phase 1b: Environment Setup & Imports
HuggingFace local cache, MPS fallback, seed locking, experiment
output directories.

### 📥 Phase 2: Data Loading, Filtering & Label Encoding
Gold layer ingestion, billable-only filter (9,660 records), full ICD-10
label encoding (1,926 classes), stratified 80/10/10 split.

### 📊 Phase 2 Observations
Label space analysis — 1,926 classes, 5 records each, ~50% val/test
coverage, ICD-10 chapter distribution.

### 📦 Phase 4: HuggingFace Dataset Construction
DatasetDict with 1,926-class ICD-10 label space locked into metadata.

### 🔍 Phase 4b: Label Space Chapter Breakdown
Distribution of 1,926 billable classes across 22 ICD-10 chapters.

### 🔡 Phase 5: Tokeniser & Dataset Tokenisation
Bio_ClinicalBERT tokeniser, 512-token max length, Assessment content
at Token 0 confirmed.

### 🧠 Phase 6: Model Initialisation
Fresh 1,926-way classification head, MPS pinning, zero-trust assertions.

### 📊 Phase 7: Trainer Configuration
Macro F1 primary metric, 20 epochs, warmup steps, MLflow confirmation.

### 🚀 Phase 8: Training Ignition (E-002)
20-epoch fine-tuning run. Best checkpoint: epoch 19.

### 📈 Phase 9: Training Dashboard Capture
Static PNG dashboard from TensorBoard event files.

### 🏆 Phase 10: Model Registry Promotion
Best weights, tokenizer, label mapping, metrics saved to registry.
MLflow run closed.

### 🔍 Phase 11: Confusion Matrix & Prediction Analysis
Test set evaluation, prediction diversity, top predicted vs actual codes.

### 🗺️ Phase 11b: Chapter-Level Confusion Analysis
22×22 chapter confusion matrix, within-chapter vs cross-chapter error
breakdown, per-chapter accuracy.

### 📖 E-002 Results: Interpretation
Full analysis of 20-epoch results, E-001 vs E-002 comparison,
coverage-adjusted Macro F1, strategic implications for E-003.

---

### 🎯 Experiment Objective

Fine-tune `emilyalsentzer/Bio_ClinicalBERT` as a 1,926-way ICD-10
classifier on 9,660 billable MedSynth records, using identical
preprocessing and training infrastructure to E-001. The only change
from E-001 is the label column — `standard_icd10` (1,926 full ICD-10
codes) instead of `icd3_label` (675 ICD-3 stems).

**Official E-002 Results (Best checkpoint: epoch 19):**

| Metric | Validation | Test |
|---|---|---|
| Macro F1 | 0.371 | 0.352 |
| Accuracy | 49.6% | 46.9% |
| Top-5 Accuracy | 77.5% | 76.8% |
| Coverage-adjusted Macro F1 | ~0.74 | ~0.70 |

**Key finding:** Chapter-level accuracy is 82.9% vs code-level accuracy
of 46.9% — 67.8% of all errors stay within the correct ICD-10 chapter.
This directly motivates a hierarchical two-stage architecture for E-003.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

# 🧠 Bio_ClinicalBERT for Full ICD-10 Classification (E-002)

This notebook implements the second modelling experiment on the MedSynth
Gold layer. Having established a strong ICD-3 baseline in E-001 (Macro F1
= 0.760, Accuracy = 82.7%, Top-5 = 91.5%), we now move to the full
ICD-10 label space — 2,037 distinct codes — to determine how much
performance the coarser ICD-3 grouping was buying us.

## 🎯 Objective

Fine-tune `emilyalsentzer/Bio_ClinicalBERT` as a 2,037-way ICD-10
classifier on the same MedSynth Gold layer used in E-001, using identical
preprocessing and training infrastructure. The only change from E-001 is
the label space — from 675 ICD-3 stems to 2,037 full ICD-10 codes.

This controlled comparison isolates the effect of label granularity and
answers the question: how much does collapsing to ICD-3 categories help,
and is the model capable of learning fine-grained ICD-10 distinctions
from ~5 training examples per code?

| Component | Strategy | Detail |
|---|---|---|
| **Model** | `emilyalsentzer/Bio_ClinicalBERT` | Same as E-001 — no architectural change |
| **Input** | `apso_note` column | APSO-recomposed, ICD-10-redacted — identical to E-001 |
| **Labels** | `standard_icd10` column | Full 2,037-code ICD-10 label space |
| **Primary metric** | Macro F1 | Equal weight to all 2,037 classes |
| **Tracking** | MLflow SQLite | Consistent with E-001 experiment tracking |
| **Hardware** | Apple Silicon MPS | Same training environment as E-001 |

## 🔬 Scientific Approach

1. **Same Gold layer, different label column.** E-001 used `icd3_label`
   derived from `standard_icd10[:3]`. E-002 uses `standard_icd10`
   directly — the canonical decimal-restored ICD-10 code produced by
   EDA Phase 2a.

2. **Same APSO-ordered, leakage-free input.** The `apso_note` column
   is unchanged from E-001 — Assessment-first, zero explicit ICD-10
   code strings, produced by EDA Phases 3a and 3c.

3. **Label space challenge.** 2,037 classes with ~5 training examples
   per class is a genuinely difficult low-resource classification task.
   Stratified splitting is not feasible at this granularity — the
   train/val/test split strategy from E-001 applies with the same
   unstratified val/test caveat.

4. **code_status filter decision.** The Gold layer contains 9,660
   billable records, 555 noisy_111 records, and 25 placeholder_x
   records. For E-002 we train on billable records only — noisy_111
   parent codes map to the same 3-character stem as their children
   at the ICD-3 level but are distinct labels at the ICD-10 level,
   and including them would introduce label ambiguity.

## 🛠️ Technical Stack

- **Model:** `emilyalsentzer/Bio_ClinicalBERT`
- **Framework:** HuggingFace Transformers + Trainer API
- **Metrics:** Macro F1 (primary), Top-1 Accuracy, Top-5 Accuracy
- **Experiment tracking:** MLflow with SQLite backend
- **Data:** MedSynth Gold layer Parquet — 9,660 billable records,
  2,037 ICD-10 classes

---

**Next:** Phase 1 — Experiment configuration and Gold layer discovery.

</div>

### Run E-002 Results

**Status:** `Complete` | **Target:** Full ICD-10 | **Best epoch:** 19

| Epoch | Train Loss | Val Loss | Accuracy | Macro F1 | Top-5 Accuracy |
|---|---|---|---|---|---|
| 1 | 7.588 | 7.573 | 0.1% | 0.000 | 0.2% |
| 5 | 6.555 | 6.455 | 8.8% | 0.052 | 25.1% |
| 10 | 5.221 | 5.246 | 35.4% | 0.249 | 65.1% |
| 15 | 4.395 | 4.593 | 46.3% | 0.342 | 75.7% |
| 19 | 4.054 | 4.387 | 49.6% | 0.371 | 77.5% |
| 20 | 4.078 | 4.379 | 49.5% | 0.371 | 77.2% |

**Test set (final evaluation):**

| Metric | Value |
|---|---|
| Macro F1 | 0.352 |
| Accuracy | 46.9% |
| Top-5 Accuracy | 76.8% |
| Loss | 4.417 |
| Chapter-level accuracy | 82.9% |
| Coverage-adjusted Macro F1 | ~0.70 |

**Runtime:** ~140 minutes at 18.5 samples/sec on Apple Silicon MPS.

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔬 Phase 1: Experiment Configuration (E-002)

Configuration-driven setup consistent with the E-001 architecture.
The only substantive changes from notebook 02 are the label scheme
(`icd10_full` instead of `icd3`) and the `code_status_filter`
(`billable` only).

### Key Controls

- **Data source:** Same Gold layer Parquet as E-001 —
  `medsynth_gold_apso_*.parquet` from `data/gold/`.
- **Records:** 9,660 billable records only (`code_status == "billable"`).
  Noisy 111 parent codes are excluded at the ICD-10 level because a
  parent code like `J18` is a distinct label from its children `J18.1`,
  `J18.9` etc., creating genuine label ambiguity that does not exist
  at the ICD-3 level where they all collapsed to the same stem.
- **Input:** `apso_note` column — APSO-recomposed, ICD-10-redacted,
  identical to E-001. No further text preprocessing applied.
- **Label scheme:** `standard_icd10` — full canonical ICD-10 code
  (e.g. `M25.562`), 2,037 distinct classes.
- **Epochs:** 20 — consistent with E-001 final run.
- **Hardware:** Apple Silicon MPS with CPU fallback enabled.
- **Tracking:** MLflow SQLite backend, consistent with E-001.

</div>

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENT CONFIGURATION (E-002 FULL ICD-10)
# ==============================================================================
import sys
import torch
import mlflow
import polars as pl
from pathlib import Path

# ------------------------------------------------------------------------------
# 1. BOOTSTRAP: DISCOVER PROJECT ROOT (enables 'import src')
# ------------------------------------------------------------------------------
print("🔍 Discovering project root...")

try:
    current = Path.cwd()
    while current != current.parent:
        if (current / "artifacts.yaml").exists():
            PROJECT_ROOT = current.resolve()
            break
        current = current.parent
    else:
        raise FileNotFoundError(
            "Could not find artifacts.yaml in current or parent directories."
        )
    print(f"   📍 Project root: .../{PROJECT_ROOT.name}")
except FileNotFoundError as e:
    print(f"❌ CRITICAL: {e}")
    raise

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"   📦 Project root added to sys.path")

# ------------------------------------------------------------------------------
# 2. IMPORT CONFIG
# ------------------------------------------------------------------------------
from src.config import config

if Path(config.project_root) != PROJECT_ROOT:
    raise RuntimeError(
        f"Project root mismatch!\n"
        f"  Bootstrap found: {PROJECT_ROOT}\n"
        f"  Config reports:  {config.project_root}"
    )

print(f"   ✅ Config loaded from: {config.config_path}")

# ------------------------------------------------------------------------------
# 3. MLFLOW SQLITE BACKEND
# ------------------------------------------------------------------------------
DB_PATH = PROJECT_ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# ------------------------------------------------------------------------------
# 4. EXPERIMENT PARAMETERS
# ------------------------------------------------------------------------------
cfg = {
    "experiment_id":       "E-002",
    "experiment_name":     "E-002_FullICD10_ClinicalBERT",
    #"description":         "Full ICD-10 classification | 2,037 classes | apso_note | billable only",
    "description":         "Full ICD-10 classification | 1,926 billable classes | apso_note | billable only",
    "model_name":          "emilyalsentzer/Bio_ClinicalBERT",
    "payload_type":        "note_only",
    "label_scheme":        "icd10_full",
    "code_status_filter":  "billable",     # billable only — noisy_111 creates label ambiguity at ICD-10 level
    "max_length":          512,
    "num_epochs":          20,             # consistent with E-001 final run
    "learning_rate":       2e-5,
    "batch_size":          16,
    "weight_decay":        0.01,
    "warmup_ratio":        0.1,            # converted to warmup_steps in Phase 7
    "use_special_tokens":  False,          # natural APSO text — no bracket markers
    "seed":                42
}

# ------------------------------------------------------------------------------
# 5. LOCATE GOLD LAYER PARQUET
# ------------------------------------------------------------------------------
gold_dir      = config.resolve_path("data", "gold")
parquet_files = sorted(gold_dir.glob("medsynth_gold_apso_*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No Gold layer Parquet found in {gold_dir}\n"
        f"Please run the EDA notebook (Phase 4) first."
    )

GOLD_PARQUET_PATH = parquet_files[-1]
print(f"\n   📁 Gold layer: {GOLD_PARQUET_PATH.name}")

# ------------------------------------------------------------------------------
# 6. MLFLOW EXPERIMENT SETUP
# ------------------------------------------------------------------------------
mlflow.set_experiment(cfg["experiment_name"])

if mlflow.active_run():
    print(f"🔄 Closing active run: {mlflow.active_run().info.run_id}")
    mlflow.end_run()

run = mlflow.start_run(run_name=f"{cfg['experiment_id']}_Full_ICD10")
mlflow.log_params(cfg)
mlflow.log_param("gold_parquet", GOLD_PARQUET_PATH.name)

# ------------------------------------------------------------------------------
# 7. DEVICE SETUP
# ------------------------------------------------------------------------------
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
mlflow.log_param("device", str(device))

# ------------------------------------------------------------------------------
# 8. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 1: Experiment Configuration",
    action="e002_config_initialised",
    details={
        "experiment_id":      cfg["experiment_id"],
        "model_name":         cfg["model_name"],
        "label_scheme":       cfg["label_scheme"],
        "code_status_filter": cfg["code_status_filter"],
        "num_epochs":         cfg["num_epochs"],
        "gold_parquet":       GOLD_PARQUET_PATH.name,
        "device":             str(device),
        "mlflow_db":          str(DB_PATH),
    },
    notebook="03-Model_ClinicalBERT_Surgical_ICD10"
)

print(f"\n🔒 Experiment locked:  {cfg['experiment_name']}")
print(f"🚀 Acceleration:       {device.type.upper()}")
print(f"📊 MLflow backend:     {DB_PATH.name}")
print(f"✅ Configuration ready for ingestion.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## ⚙️ Phase 1b: Environment Setup & Imports

One-time environment configuration identical to notebook 02. Sets up the
HuggingFace local cache, MPS fallback, reproducibility seed, and experiment
output directories before any data loading or model initialisation begins.

All downstream phases depend on `HF_CACHE_DIR`, `EXP_DIR`, `CHECKPOINT_DIR`,
and `TENSORBOARD_DIR` being defined here.

</div>

In [ ]:
# ==============================================================================
# PHASE 1b: ENVIRONMENT SETUP & IMPORTS
# ==============================================================================

import os
import numpy as np
from pathlib import Path
from transformers import set_seed

# Integrity checks
assert "cfg" in globals(), \
    "❌ cfg not found. Run Phase 1 first."
assert "PROJECT_ROOT" in globals(), \
    "❌ PROJECT_ROOT not found. Run Phase 1 first."
assert "GOLD_PARQUET_PATH" in globals(), \
    "❌ GOLD_PARQUET_PATH not found. Run Phase 1 first."

# HuggingFace local cache
HF_CACHE_DIR = PROJECT_ROOT / "data" / "cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"]               = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"]          = str(HF_CACHE_DIR)
os.environ["HF_HUB_READ_TIMEOUT"]   = "120"
os.environ["HF_HUB_CONNECT_TIMEOUT"] = "60"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Imports
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed
)

# Seed
active_seed = cfg.get("seed", 42)
set_seed(active_seed)

# Experiment output directory
EXP_DIR      = config.resolve_path("outputs", "evaluations") / cfg["experiment_name"]
CHECKPOINT_DIR  = EXP_DIR / "checkpoints"
TENSORBOARD_DIR = EXP_DIR / "tensorboard"
EXP_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
TENSORBOARD_DIR.mkdir(parents=True, exist_ok=True)

os.environ["TENSORBOARD_LOGGING_DIR"] = str(TENSORBOARD_DIR)

print(f"✅ Phase 1b complete")
print(f"   HF cache:        {HF_CACHE_DIR}")
print(f"   Experiment dir:  {EXP_DIR}")
print(f"   Seed:            {active_seed}")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📥 Phase 2: Data Loading, Filtering & Label Encoding (E-002)

Loads the Gold layer Parquet, filters to billable records only, encodes
the full 2,037-class ICD-10 label space, and produces train/val/test splits.

### Key Differences from E-001

**Billable filter applied.** At the ICD-10 level, noisy_111 parent codes
(e.g. `J18`) are distinct labels from their children (`J18.1`, `J18.9`).
Including them would introduce label ambiguity that did not exist at the
ICD-3 level where all codes sharing the same 3-character prefix collapsed
to a single stem. E-002 therefore restricts to the 9,660 billable records.

**Label column is `standard_icd10` not `icd3_label`.** The full canonical
ICD-10 code is used directly — no stem truncation applied.

**Label space is 2,037 classes with ~5 records each.** At this density,
the 20% temp pool contains only 1 record per class for most codes — below
sklearn's minimum of 2 for stratified splitting. The same approach as
E-001 applies: stratified train/temp split, random val/test split.

### Expected Coverage Gap

At ~5 records per ICD-10 code, the val and test sets will have
significantly lower class coverage than E-001's ICD-3 results (76.3%
and 72.4% respectively). The Macro F1 reported for E-002 will be an
even more conservative lower bound than E-001's figures.

</div>

In [ ]:
# ==============================================================================
# PHASE 2: DATA LOADING, FILTERING & LABEL ENCODING (E-002 FULL ICD-10)
# ==============================================================================
# Purpose: Load the Gold layer Parquet, filter to billable records only,
# encode the full ICD-10 label space, and produce train/val/test splits.
#
# Key differences from E-001 (notebook 02):
#   - Label column: standard_icd10 (full ICD-10) not icd3_label (3-char stem)
#   - Label space: 2,037 classes not 675
#   - Records: 9,660 billable only (code_status == "billable")
#   - Stratification: same approach — stratified train/temp, random val/test
#     (even fewer records per class at ICD-10 level makes stratification
#     of the full 3-way split impossible)
# ==============================================================================

import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------------------
# 1. LOAD GOLD LAYER
# ------------------------------------------------------------------------------
print(f"📂 Loading Gold layer: {GOLD_PARQUET_PATH.name}")

df_gold = pl.read_parquet(GOLD_PARQUET_PATH)

print(f"   ✅ Loaded: {len(df_gold):,} records, {len(df_gold.columns)} columns")

# Confirm required columns
required_cols = {'id', 'apso_note', 'standard_icd10', 'code_status'}
missing_cols  = required_cols - set(df_gold.columns)
if missing_cols:
    raise ValueError(
        f"❌ Gold layer missing required columns: {missing_cols}\n"
        f"   Please re-run the EDA notebook through Phase 3c."
    )
print(f"   ✅ All required columns present")

# ------------------------------------------------------------------------------
# 2. FILTER TO BILLABLE RECORDS ONLY
# ------------------------------------------------------------------------------
# At the ICD-10 level, noisy_111 parent codes (e.g. J18) are distinct labels
# from their children (J18.1, J18.9). Including them would introduce label
# ambiguity not present at the ICD-3 level where they collapsed to the same
# stem. We therefore restrict to billable records for E-002.
# ------------------------------------------------------------------------------
print(f"\n📊 Full dataset code_status breakdown:")
for row in (df_gold.group_by("code_status")
            .agg(pl.len().alias("count"))
            .sort("count", descending=True)
            .iter_rows(named=True)):
    pct = row['count'] / len(df_gold) * 100
    print(f"   {row['code_status']:20s}: {row['count']:,} ({pct:.1f}%)")

df_gold = df_gold.filter(pl.col("code_status") == cfg["code_status_filter"])
print(f"\n   ✅ Filtered to '{cfg['code_status_filter']}': {len(df_gold):,} records")

mlflow.log_param("filtered_record_count", len(df_gold))

# ------------------------------------------------------------------------------
# 3. LABEL ENCODING
# ------------------------------------------------------------------------------
# Build a deterministic alphabetically-sorted string → integer mapping.
# This ensures reproducibility across runs and notebooks.
# ------------------------------------------------------------------------------
print(f"\n🔧 Building ICD-10 label encoder...")

all_labels = sorted(df_gold["standard_icd10"].unique().to_list())
label2id   = {label: idx for idx, label in enumerate(all_labels)}
id2label   = {idx: label for label, idx in label2id.items()}
num_labels = len(label2id)

df_gold = df_gold.with_columns(
    pl.col("standard_icd10")
    .replace(list(label2id.keys()), [label2id[k] for k in label2id.keys()])
    .cast(pl.Int64)
    .alias("label_id")
)

print(f"   ✅ Label encoder built: {num_labels:,} ICD-10 classes")
print(f"   📊 Label ID range: 0 – {num_labels - 1}")
print(f"   📊 Sample labels: {all_labels[:5]} ... {all_labels[-3:]}")

cfg["num_labels"] = num_labels
mlflow.log_param("num_labels", num_labels)

# Frequency distribution
freq_dist = (df_gold.group_by("standard_icd10")
             .agg(pl.len().alias("count"))
             .group_by("count")
             .agg(pl.len().alias("num_codes"))
             .sort("count"))

print(f"\n   📊 ICD-10 frequency distribution (sample):")
for row in freq_dist.head(5).iter_rows(named=True):
    print(f"      {row['num_codes']:4d} codes with exactly {row['count']:2d} record(s)")

# ------------------------------------------------------------------------------
# 4. STRUCTURAL CHECK
# ------------------------------------------------------------------------------
print(f"\n🔍 Structural check...")

null_notes  = df_gold.filter(
    pl.col("apso_note").is_null() | (pl.col("apso_note") == "")
).height
null_labels = df_gold.filter(pl.col("label_id").is_null()).height

if null_notes > 0 or null_labels > 0:
    print(f"   ⚠️  {null_notes:,} null/empty apso_note values")
    print(f"   ⚠️  {null_labels:,} null label_id values — dropping")
    df_gold = df_gold.filter(
        pl.col("apso_note").is_not_null() &
        (pl.col("apso_note") != "") &
        pl.col("label_id").is_not_null()
    )
    print(f"   ✅ Records after drop: {len(df_gold):,}")
else:
    print(f"   ✅ No null values — all {len(df_gold):,} records ready for splitting")

# ------------------------------------------------------------------------------
# 5. TRAIN / VAL / TEST SPLIT (80 / 10 / 10)
# ------------------------------------------------------------------------------
# Stratification note:
#   At the ICD-10 level (~5 records per code), the 20% temp pool contains
#   only 1 record per code for most classes — below sklearn's minimum of 2
#   for stratified splitting. The same approach as E-001 applies:
#   - Train/temp split: STRATIFIED by label_id
#   - Val/test split: RANDOM (not stratified)
# ------------------------------------------------------------------------------
print(f"\n✂️  Splitting dataset (80/10/10)...")

df_pd = df_gold.select([
    "id", "apso_note", "label_id", "standard_icd10", "code_status"
]).to_pandas()

train_df, temp_df = train_test_split(
    df_pd,
    test_size=0.2,
    stratify=df_pd["label_id"],
    random_state=cfg["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=None,       # not stratified — see note above
    random_state=cfg["seed"]
)

print(f"   ✅ Split complete:")
print(f"      Train: {len(train_df):,} records ({len(train_df)/len(df_pd)*100:.1f}%) — stratified")
print(f"      Val:   {len(val_df):,} records  ({len(val_df)/len(df_pd)*100:.1f}%) — random")
print(f"      Test:  {len(test_df):,} records  ({len(test_df)/len(df_pd)*100:.1f}%) — random")

mlflow.log_params({
    "train_size":          len(train_df),
    "val_size":            len(val_df),
    "test_size":           len(test_df),
    "icd10_classes":       num_labels,
    "train_stratified":    True,
    "val_test_stratified": False,
})

# ------------------------------------------------------------------------------
# 6. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 2: Data Loading & Splitting",
    action="data_loaded_and_split",
    details={
        "gold_parquet":        GOLD_PARQUET_PATH.name,
        "total_records":       len(df_gold),
        "code_status_filter":  cfg["code_status_filter"],
        "icd10_classes":       num_labels,
        "train_size":          len(train_df),
        "val_size":            len(val_df),
        "test_size":           len(test_df),
        "null_notes_dropped":  null_notes,
        "null_labels_dropped": null_labels,
        "train_stratified":    True,
        "val_test_stratified": False,
        "stratification_note": (
            "Val/test split not stratified — MedSynth uniform 5-per-code "
            "sampling produces only 1 record per class in the 20% temp pool "
            "at the ICD-10 level. Train/temp split remains stratified."
        )
    },
    notebook="03-Model_ClinicalBERT_Surgical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 2 complete: {num_labels:,}-way ICD-10 classification task ready")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

### 📊 Phase 2 Observations

**1,926 ICD-10 classes from 9,660 billable records.** The 111-class
reduction from the full dataset's 2,037 codes reflects the noisy_111
and placeholder_x records that were excluded — those codes only appeared
in non-billable records and are therefore absent from the billable-only
training set.

**The frequency distribution is the starkest possible low-resource
scenario.** 1,920 of 1,926 classes (99.7%) have exactly 5 records.
Only 6 classes have 10 records. This means the training set provides
approximately 4 examples per class after the 80/20 train/temp split.

This is a fundamentally different challenge from E-001:

| Factor | E-001 (ICD-3) | E-002 (ICD-10) |
|---|---|---|
| Classes | 675 | 1,926 |
| Avg records/class | ~15 | ~5 |
| Train examples/class | ~12 | ~4 |
| Val/test class coverage | ~75% | ~50% |
| Random classifier Macro F1 | ~0.0015 | ~0.0005 |

**Val/test class coverage is ~50%.** With only 5 records per ICD-10 code
and a random val/test split, approximately half the 1,926 classes are
absent from each evaluation split — confirmed by the Phase 2 verification
output (val: 964/1,926 classes, test: 966/1,926 classes). Absent classes
contribute F1 = 0.0 to the macro average, meaning the reported Macro F1
is approximately half the true Macro F1 on seen classes.

Direct comparison with E-001's Macro F1 figures should account for this
difference — E-001 had ~75% val/test coverage vs E-002's ~50%. A fairer
cross-experiment comparison uses coverage-adjusted Macro F1:
adjusted = reported_macro_f1 × (total_classes / classes_present).

**The E-002 Macro F1 should be interpreted as a conservative lower bound.**
A meaningful result is not necessarily a high absolute Macro F1 — it is
evidence that the model has learned genuine ICD-10 diagnostic signal
rather than defaulting to high-frequency classes.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📦 Phase 4: HuggingFace Dataset Construction

Wraps the validated splits into a HuggingFace `DatasetDict` with the
1,926-class ICD-10 label space locked into the dataset metadata.
Identical approach to E-001 — only the label space size differs.

The label encoding audit at the end of this cell confirms the class
coverage figures discussed in Phase 2: train has 100% coverage
(stratified split), val and test have approximately 50% coverage
(random split — expected given 5 records per ICD-10 code).

</div>

In [ ]:
# ==============================================================================
# PHASE 4: HUGGING FACE DATASET CONSTRUCTION (E-002)
# ==============================================================================

from datasets import Dataset, DatasetDict, Features, Value, ClassLabel

# ------------------------------------------------------------------------------
# 1. DEFINE FEATURES — locks the 1,926-way label space into dataset metadata
# ------------------------------------------------------------------------------
class_names = sorted(list(label2id.keys()))

features = Features({
    'text':  Value('string'),
    'label': ClassLabel(names=class_names)
})

# ------------------------------------------------------------------------------
# 2. BUILD DATASETDICT
# apso_note → text, label_id → label
# ------------------------------------------------------------------------------
raw_datasets = DatasetDict({
    "train": Dataset.from_dict(
        {"text": train_df["apso_note"].tolist(),
         "label": train_df["label_id"].tolist()},
        features=features
    ),
    "val": Dataset.from_dict(
        {"text": val_df["apso_note"].tolist(),
         "label": val_df["label_id"].tolist()},
        features=features
    ),
    "test": Dataset.from_dict(
        {"text": test_df["apso_note"].tolist(),
         "label": test_df["label_id"].tolist()},
        features=features
    ),
})

# ------------------------------------------------------------------------------
# 3. VERIFY
# ------------------------------------------------------------------------------
print(f"✅ DatasetDict constructed")
print(f"   {'Split':8s}  {'Records':>8s}  {'Classes':>8s}")
print(f"   {'─'*30}")
for split in ["train", "val", "test"]:
    n_classes = raw_datasets[split].features['label'].num_classes
    print(f"   {split:8s}  {len(raw_datasets[split]):>8,}  {n_classes:>8,}")

print(f"\n   ✅ Label space locked: {num_labels:,} ICD-10 classes")

# Spot-check
sample = raw_datasets["train"][0]
print(f"\n👁️  Sample training record:")
print(f"   Label ID:  {sample['label']}")
print(f"   Label str: {id2label[sample['label']]}")
print(f"   Text:      {sample['text'][:150]}...")

# ------------------------------------------------------------------------------
# 4. LABEL ENCODING AUDIT
# ------------------------------------------------------------------------------
print(f"\n🕵️  Label encoding audit...")

for split in ["train", "val", "test"]:
    unique_in_split = len(set(raw_datasets[split]["label"]))
    coverage_pct    = unique_in_split / num_labels * 100
    print(f"   📊 {split:5s}: {len(raw_datasets[split]):>6,} records, "
          f"{unique_in_split:>5,} / {num_labels:,} classes ({coverage_pct:.1f}%)")

# ------------------------------------------------------------------------------
# 5. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 4: HuggingFace Dataset",
    action="dataset_dict_constructed",
    details={
        "train_size":  len(raw_datasets["train"]),
        "val_size":    len(raw_datasets["val"]),
        "test_size":   len(raw_datasets["test"]),
        "num_classes": num_labels,
        "text_column": "apso_note",
        "label_column": "standard_icd10",
    },
    notebook="03-Model_ClinicalBERT_Surgical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 4 complete: DatasetDict ready for tokenisation")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 4b: Label Space Chapter Breakdown

Before tokenisation begins, we audit the distribution of the 1,926 ICD-10
classes across clinical chapters. This gives a structural picture of what
the model is being asked to learn and which chapters are likely to be the
primary sources of confusion.

All 1,926 classes are confirmed billable CDC FY2026 leaf codes. The
Noisy 111 parent codes identified in the EDA were excluded via the
`code_status == "billable"` filter in Phase 2 — no further audit is
required here.

</div>

In [ ]:
# ==============================================================================
# PHASE 4b: LABEL SPACE CHAPTER BREAKDOWN
# ==============================================================================
# Shows which ICD-10 chapters are represented in the 1,926-class label space.
# All codes are confirmed billable CDC FY2026 leaf codes — no Noisy 111 present.
# ==============================================================================

from collections import Counter

chapter_counts = Counter(code[0] for code in label2id.keys())

print(f"📊 ICD-10 Chapter Distribution ({num_labels:,} billable classes):")
print(f"   {'Chapter':8s}  {'Classes':>8s}  {'% of total':>10s}")
print(f"   {'─'*32}")
for chapter, count in sorted(chapter_counts.items()):
    pct = count / num_labels * 100
    print(f"   {chapter:8s}  {count:>8,}  {pct:>9.1f}%")

print(f"\n   ✅ All {num_labels:,} classes are confirmed billable CDC FY2026 leaf codes")
print(f"   ✅ Noisy 111 parent codes excluded via code_status filter in Phase 2")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

### 📊 ICD-10 Chapter Distribution

The 1,926 billable classes span 22 ICD-10 chapters. Three chapters
dominate the label space:

- **Z-codes (13.7%, 263 classes)** — Administrative and contextual codes
  covering routine examinations, screenings, follow-up care, and social
  determinants. Z-codes share similar clinical note language across many
  subclasses, making them a likely source of within-chapter confusion.

- **M-codes (11.5%, 222 classes)** — Musculoskeletal conditions. Consistent
  with the MedSynth IQVIA disease distribution and E-001 findings where
  M25 was the most frequent training category.

- **R-codes (10.2%, 196 classes)** — Symptom and sign codes. These describe
  clinical presentations without specifying a definitive aetiology — by
  design they share clinical language with the definitive diagnosis codes
  they often accompany, making them a natural source of semantic confusion.

At the other extreme, P-codes (5 classes), Q-codes (4 classes), and
U-codes (1 class) are so rare in the label space that the model will
have very few training examples to learn from. These chapters will
contribute disproportionately to Macro F1 suppression.

All 1,926 classes are confirmed billable CDC FY2026 leaf codes. The
Noisy 111 parent codes identified in the EDA were excluded via the
`code_status == "billable"` filter in Phase 2.

</div>

# Here prob

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔡 Phase 5: Tokeniser & Dataset Tokenisation (E-002)

Identical approach to E-001. The Bio_ClinicalBERT tokenizer is used
without special tokens — natural APSO text with Assessment content
at Token 0. The only difference from E-001 is the label space size
(1,926 ICD-10 classes vs 675 ICD-3 categories), which does not affect
tokenisation.

</div>

In [ ]:
# ==============================================================================
# PHASE 5: TOKENISER & DATASET TOKENISATION (E-002)
# ==============================================================================

from transformers import AutoTokenizer

print(f"📥 Loading tokenizer: {cfg['model_name']}")

tokenizer = AutoTokenizer.from_pretrained(
    cfg["model_name"],
    cache_dir=str(HF_CACHE_DIR)
)

print(f"   ✅ Tokenizer loaded")
print(f"   📏 Vocabulary size: {len(tokenizer):,}")

# use_special_tokens = False — natural APSO text, no bracket markers
cfg["use_special_tokens"] = False

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=cfg["max_length"]
    )

print(f"\n🔄 Tokenising dataset (max_length={cfg['max_length']})...")

tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["text"]
)

print(f"   ✅ Tokenisation complete")
print(f"   📊 Columns: {tokenized_datasets['train'].column_names}")

# Token 0 verification
sample_ids    = tokenized_datasets["train"][0]["input_ids"]
first_tokens  = tokenizer.decode(sample_ids[:10], skip_special_tokens=False)
first_content = tokenizer.decode(sample_ids[1:15], skip_special_tokens=True)

print(f"\n🔬 Token verification (first training record):")
print(f"   Tokens 0–9:   {first_tokens}")
print(f"   Content 1–14: {first_content}")

tokenized_datasets.set_format("torch")
print(f"\n   ✅ Dataset format set to torch tensors")

config.log_event(
    phase="Phase 5: Tokenisation",
    action="dataset_tokenised",
    details={
        "model_name":         cfg["model_name"],
        "vocab_size":         len(tokenizer),
        "max_length":         cfg["max_length"],
        "use_special_tokens": False,
        "train_size":         len(tokenized_datasets["train"]),
        "val_size":           len(tokenized_datasets["val"]),
        "test_size":          len(tokenized_datasets["test"]),
    },
    notebook="03-Model_ClinicalBERT_Surgical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 5 complete: {len(tokenized_datasets['train']):,} training samples tokenised")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🧠 Phase 6: Model Initialisation (E-002)

Identical approach to E-001. The only difference is the classification
head size — 1,926 neurons for ICD-10 instead of 675 for ICD-3. The
same `ignore_mismatched_sizes=True` flag handles the pretrained head
swap, and the same UNEXPECTED/MISSING key warnings in the load report
are expected and benign.

</div>

In [ ]:
# ==============================================================================
# PHASE 6: MODEL INITIALISATION & CHECKPOINT RESOLUTION (E-002)
# ==============================================================================

from transformers import AutoModelForSequenceClassification

# ------------------------------------------------------------------------------
# 1. CHECKPOINT RESOLUTION
# ------------------------------------------------------------------------------
RESUME         = cfg.get("resume_from_checkpoint", False)
checkpoint_dir = CHECKPOINT_DIR

# ------------------------------------------------------------------------------
# 2. LOAD MODEL
# ------------------------------------------------------------------------------
if RESUME and checkpoint_dir.exists():
    print(f"🔄 Attempting to resume from checkpoint: {checkpoint_dir}")
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint_dir,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True
        )
        print("   ✅ Resumed from checkpoint successfully.")
    except Exception as e:
        print(f"   ⚠️ Checkpoint load failed: {e}")
        print(f"   ↳ Falling back to fresh initialisation.")
        RESUME = False

if not RESUME:
    print(f"🆕 Initialising fresh Bio_ClinicalBERT for: {cfg['experiment_name']}")
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["model_name"],
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
        cache_dir=str(HF_CACHE_DIR)
    )
    print(f"   ✅ Fresh model initialised.")

# ------------------------------------------------------------------------------
# 3. PIN TO DEVICE
# ------------------------------------------------------------------------------
model.to(device)
print(f"   ✅ Model pinned to: {next(model.parameters()).device}")

# ------------------------------------------------------------------------------
# 4. DATASET FORMAT
# ------------------------------------------------------------------------------
tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "label"]
)
print(f"   ✅ Dataset format set: {['input_ids', 'attention_mask', 'token_type_ids', 'label']}")

# ------------------------------------------------------------------------------
# 5. ARCHITECTURE AUDIT
# ------------------------------------------------------------------------------
print(f"\n🔍 Architecture Audit:")
print(f"   Classifier output dim: {model.num_labels:,}")
print(f"   Embedding vocab size:  {model.get_input_embeddings().weight.shape[0]:,}")
print(f"   Hardware device:       {next(model.parameters()).device}")
print(f"   Total parameters:      {sum(p.numel() for p in model.parameters()):,}")

assert model.num_labels == num_labels, \
    f"❌ Label mismatch: model has {model.num_labels}, expected {num_labels}"
assert model.get_input_embeddings().weight.shape[0] == len(tokenizer), \
    f"❌ Vocab mismatch: model={model.get_input_embeddings().weight.shape[0]}, tokenizer={len(tokenizer)}"

print(f"\n   ✅ All assertions passed")

# ------------------------------------------------------------------------------
# 6. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 6: Model Initialisation",
    action="model_initialised",
    details={
        "model_name":  cfg["model_name"],
        "num_labels":  num_labels,
        "vocab_size":  model.get_input_embeddings().weight.shape[0],
        "device":      str(next(model.parameters()).device),
        "resumed":     RESUME,
        "total_params": sum(p.numel() for p in model.parameters()),
    },
    notebook="03-Model_ClinicalBERT_Surgical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"🚀 Phase 6 complete: model ready for training on {device}")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 7: Trainer Configuration (E-002)

Identical approach to E-001. Uses `hf_compute_metrics` from
`src/evaluation.py` — accuracy, Macro F1, and Top-5 Accuracy.
Macro F1 is the primary checkpoint selection metric, equal-weighting
all 1,926 ICD-10 classes regardless of frequency.

`save_total_limit = 3` retains only the 3 best checkpoints across
20 epochs — important for disk management at this scale.

</div>

In [ ]:
# ==============================================================================
# PHASE 7: TRAINER CONFIGURATION (E-002)
# ==============================================================================

from transformers import TrainingArguments, DataCollatorWithPadding
import os

# ------------------------------------------------------------------------------
# 1. IMPORT METRICS FROM src/evaluation.py
# ------------------------------------------------------------------------------
try:
    from src.evaluation import hf_compute_metrics
    compute_metrics = hf_compute_metrics
    print("✅ hf_compute_metrics imported from src.evaluation")
    print("   Metrics: accuracy, macro_f1, top_5_accuracy")
except ImportError as e:
    raise ImportError(
        f"❌ Could not import hf_compute_metrics from src.evaluation.\n"
        f"   Error: {e}"
    )

# ------------------------------------------------------------------------------
# 2. COMPUTE WARMUP STEPS
# warmup_ratio → warmup_steps (avoids HuggingFace v5.2 deprecation warning)
# ------------------------------------------------------------------------------
total_training_steps = (
    len(tokenized_datasets["train"]) // cfg["batch_size"]
) * cfg["num_epochs"]

warmup_steps = int(cfg.get("warmup_ratio", 0.1) * total_training_steps)

print(f"\n   📊 Total training steps: {total_training_steps:,}")
print(f"   📊 Warmup steps:         {warmup_steps:,}")

# ------------------------------------------------------------------------------
# 3. TRAINING ARGUMENTS
# ------------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir              = str(CHECKPOINT_DIR),

    eval_strategy           = "epoch",
    save_strategy           = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "macro_f1",
    greater_is_better       = True,
    save_total_limit        = 3,

    num_train_epochs            = cfg["num_epochs"],
    per_device_train_batch_size = cfg["batch_size"],
    learning_rate               = cfg["learning_rate"],
    weight_decay                = cfg["weight_decay"],
    warmup_steps                = warmup_steps,

    logging_steps   = 20,
    report_to       = ["tensorboard", "mlflow"],
    seed            = cfg["seed"],

    fp16                  = False,
    dataloader_pin_memory = False,
)

# ------------------------------------------------------------------------------
# 4. INITIALISE TRAINER
# ------------------------------------------------------------------------------
trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = tokenized_datasets["train"],
    eval_dataset     = tokenized_datasets["val"],
    processing_class = tokenizer,
    data_collator    = DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics  = compute_metrics,
)

print(f"\n✅ Trainer initialised")
print(f"   Train samples:   {len(tokenized_datasets['train']):,}")
print(f"   Val samples:     {len(tokenized_datasets['val']):,}")
print(f"   Epochs:          {cfg['num_epochs']}")
print(f"   Batch size:      {cfg['batch_size']}")
print(f"   Learning rate:   {cfg['learning_rate']}")
print(f"   Warmup steps:    {warmup_steps:,}")
print(f"   Primary metric:  macro_f1")
print(f"   Label space:     {num_labels:,} ICD-10 classes")

# ------------------------------------------------------------------------------
# 5. CONFIRM ACTIVE MLFLOW RUN
# ------------------------------------------------------------------------------
active_run = mlflow.active_run()
if active_run is None:
    print("\n⚠️  WARNING: No active MLflow run.")
else:
    print(f"\n✅ Active MLflow run: {active_run.info.run_id}")
    mlflow.log_params({
        "checkpoint_dir":       str(CHECKPOINT_DIR),
        "tensorboard_dir":      str(TENSORBOARD_DIR),
        "metric_primary":       "macro_f1",
        "eval_strategy":        "epoch",
        "total_training_steps": total_training_steps,
        "warmup_steps":         warmup_steps,
    })

# ------------------------------------------------------------------------------
# 6. MONITORING
# ------------------------------------------------------------------------------
print(f"\n{'='*70}")
print(f"📈 TENSORBOARD:")
print(f"   tensorboard --logdir '{TENSORBOARD_DIR}' --port 6006")
print(f"\n📊 MLFLOW UI:")
print(f"   mlflow ui --backend-store-uri sqlite:///{PROJECT_ROOT}/mlflow.db --port 5001")
print(f"{'='*70}")

# ------------------------------------------------------------------------------
# 7. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 7: Trainer Configuration",
    action="trainer_initialised",
    details={
        "num_epochs":           cfg["num_epochs"],
        "batch_size":           cfg["batch_size"],
        "learning_rate":        cfg["learning_rate"],
        "warmup_steps":         warmup_steps,
        "total_training_steps": total_training_steps,
        "num_labels":           num_labels,
        "metric_primary":       "macro_f1",
        "mlflow_run_id":        active_run.info.run_id if active_run else None,
    },
    notebook="03-Model_ClinicalBERT_Surgical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 7 complete: trainer ready — call trainer.train() to begin")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🚀 Phase 8: Training Ignition (E-002 Full ICD-10)

Executes the 20-epoch training run on 7,728 records across 1,926 ICD-10
classes. Identical approach to E-001 — Macro F1 is the checkpoint
selection criterion, best model auto-reloaded at completion.

**Expected duration:** approximately 130–150 minutes on Apple Silicon MPS.

**Expected behaviour:** Given only ~4 training examples per class, early
epochs will show near-zero Macro F1. Meaningful improvement is expected
from epoch 5 onwards, following the same pattern observed in E-001.

> **Note:** With ~50% val/test class coverage, the reported Macro F1
> will be approximately half the true Macro F1 on seen classes. Monitor
> accuracy and Top-5 accuracy alongside Macro F1 for a complete picture.

</div>

In [ ]:
# ==============================================================================
# PHASE 8: TRAINING IGNITION (E-002 FULL ICD-10)
# ==============================================================================

import sys

if 'trainer' not in globals():
    print("❌ ERROR: trainer not found. Please run Phase 7 first.")
    sys.exit()

if mlflow.active_run() is None:
    print("⚠️  WARNING: No active MLflow run. Metrics will not be logged.")

# ------------------------------------------------------------------------------
# TRAINING SUMMARY
# ------------------------------------------------------------------------------
print(f"🚀 Igniting E-002 Full ICD-10 Training")
print(f"   Model:          {cfg['model_name']}")
print(f"   Label scheme:   Full ICD-10 ({num_labels:,} classes)")
print(f"   Epochs:         {cfg['num_epochs']}")
print(f"   Train samples:  {len(tokenized_datasets['train']):,}")
print(f"   Val samples:    {len(tokenized_datasets['val']):,}")
print(f"   Warmup steps:   {trainer.args.warmup_steps:,}")
print(f"   Device:         {device}")
print(f"   Checkpoints:    {CHECKPOINT_DIR}")

# ------------------------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------------------------
train_result = trainer.train()

# ------------------------------------------------------------------------------
# LOG TRAINING RESULTS
# ------------------------------------------------------------------------------
print(f"\n✅ Training complete")
print(f"   Training loss:  {train_result.training_loss:.4f}")
print(f"   Runtime:        {train_result.metrics.get('train_runtime', 0):.1f}s")
print(f"   Samples/sec:    {train_result.metrics.get('train_samples_per_second', 0):.1f}")

mlflow.log_metrics({
    "final_train_loss":         train_result.training_loss,
    "train_runtime_s":          train_result.metrics.get("train_runtime", 0),
    "train_samples_per_second": train_result.metrics.get("train_samples_per_second", 0),
})

config.log_event(
    phase="Phase 8: Training",
    action="training_complete",
    details={
        "training_loss": train_result.training_loss,
        "train_runtime": train_result.metrics.get("train_runtime", 0),
        "num_epochs":    cfg["num_epochs"],
        "train_samples": len(tokenized_datasets["train"]),
        "num_labels":    num_labels,
    },
    notebook="03-Model_ClinicalBERT_Surgical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 8 complete")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 E-002 Results: Interpretation (20-Epoch Full ICD-10 Run)

### Final Performance (20 Epochs, 1,926 ICD-10 Classes)

| Metric | Value | Context |
|---|---|---|
| Macro F1 | 0.371 | Primary metric — equal weight to all 1,926 classes |
| Accuracy | 49.6% | Top-1 classification across 1,926 ICD-10 codes |
| Top-5 Accuracy | 77.5% | Correct code in model's top 5 predictions |
| Final val loss | 4.387 | Still decreasing — model not fully converged |
| Best epoch | 19 | Selected by peak Macro F1, auto-reloaded |
| Runtime | ~140 min | 18.5 samples/sec on Apple Silicon MPS |

---

### What the Numbers Mean

**Macro F1 of 0.371** on a 1,926-way classification task with approximately
4 training examples per class is a strong result. A random classifier across
1,926 classes would achieve a Macro F1 of approximately 0.0005 — the trained
model is 740x above chance. The model has learned genuine ICD-10 diagnostic
signal at the full code level, not just broad chapter-level patterns.

**Top-5 Accuracy of 77.5%** is the most clinically meaningful figure. In a
real coding assistance workflow, surfacing the correct ICD-10 code in the
top 5 suggestions 77.5% of the time would substantially reduce coder effort.
A human coder could select from a short ranked list rather than navigating
the full 2,037-code ICD-10 hierarchy.

**Accuracy of 49.6%** on a 1,926-way task with ~4 training examples per
class is remarkable. For comparison, a majority-class classifier would
achieve approximately 0.05% accuracy (1 out of 1,926 classes).

---

### TensorBoard Curve Analysis

**eval/accuracy** — clean S-curve from ~0% to ~50%, still rising at step
9,660. The curve has not flattened, confirming the model has not exhausted
its capacity. ✅

**eval/loss** — steady decrease from ~7.6 to ~4.4 with no divergence from
train loss until the final few epochs. The train/val gap at epoch 20
(4.078 vs 4.379 = 0.301) is modest — mild overfitting beginning in the
final 3 epochs but not catastrophic. ✅

**eval/macro_f1** — rising curve reaching ~0.30 smoothed (~0.371 actual).
Still trending upward at step 9,660 — unlike E-001 which showed a clear
plateau, E-002 has not reached its natural ceiling at 20 epochs. ✅

**eval/top_5_accuracy** — strong upward curve reaching ~0.80 smoothed
(~0.775 actual), still rising. ✅

**train/learning_rate** — correct warmup/decay schedule. Rises to 2e-5
over the first 966 steps then decays linearly to 0 at step 9,660. The
learning rate reaching zero at epoch 20 is what stopped improvement,
not model capacity. ✅

**train/grad_norm** — higher variance than E-001 throughout, reflecting
the more difficult optimisation landscape of the 1,926-way label space.
Stabilises in the final epochs as the learning rate decays. Expected. ✅

---

### Convergence Analysis

Unlike E-001 which showed a clear performance plateau between epochs 19
and 20, E-002 metrics are still improving at epoch 20. Three signals
support this:

1. **Val loss still decreasing** — 4.387 at epoch 20 vs 4.386 at epoch
   19, essentially flat but not diverging.
2. **Macro F1 improvement continues** — 0.342 at epoch 15 → 0.371 at
   epoch 19, a meaningful 2.9pp gain in the final 4 epochs.
3. **No overfitting plateau** — the train/val loss gap is widening slowly
   but val metrics are still improving, indicating the model is still
   in the productive training window.

A 25–30 epoch run would likely push Macro F1 above 0.40 and accuracy
above 55%. The model has not found its natural ceiling.

---

### The Critical Comparison: E-001 vs E-002

| Metric | E-001 (ICD-3) | E-002 (ICD-10) | Raw gap |
|---|---|---|---|
| Macro F1 | 0.760 | 0.371 | -0.389 |
| Accuracy | 82.7% | 49.6% | -33.1pp |
| Top-5 Accuracy | 91.5% | 77.5% | -14.0pp |
| Classes | 675 | 1,926 | 2.85x more |
| Train examples/class | ~12 | ~4 | 3x fewer |
| Val class coverage | ~75% | ~50% | -25pp |

**The raw comparison overstates the performance gap.** Two structural
differences inflate the apparent E-002 disadvantage:

1. **3x fewer training examples per class** — E-001 had ~12 training
   examples per ICD-3 category vs E-002's ~4 per ICD-10 code. All else
   equal, 3x fewer examples produces meaningfully lower metrics.

2. **Val/test coverage gap** — E-001's val set covered ~75% of classes
   while E-002's covers only ~50%. Classes absent from the val set
   contribute F1 = 0.0 to the macro average. Coverage-adjusting E-002's
   Macro F1: 0.371 × (1,926 / 966) ≈ **0.74** — remarkably close to
   E-001's 0.760.

This coverage-adjusted comparison suggests the model's **per-class
learning efficiency is essentially the same at both granularities**.
The model learns ICD-10 codes as effectively as ICD-3 categories when
controlling for data volume and evaluation coverage. The raw performance
gap is a measurement artefact, not a fundamental capability difference.

---

### Known Limitations

1. **~50% val/test coverage** — 962 of 1,926 classes are absent from
   the val set. The reported Macro F1 is a conservative lower bound.
   Coverage-adjusted Macro F1 ≈ 0.74.

2. **Uniform sampling bias** — MedSynth's 5-per-code design does not
   reflect real-world ICD-10 frequency distributions where a small
   number of codes dominate clinical volume.

3. **Model not converged** — unlike E-001, E-002 has not found its
   natural ceiling at 20 epochs. Extended training is warranted.

---

### Recommendations for Next Experiments

| Experiment | Change | Expected Impact |
|---|---|---|
| E-002b | Extend to 30 epochs | Macro F1 likely > 0.40 |
| E-003 | Hierarchical two-stage (ICD-3 router → ICD-10 resolver) | Leverages E-001 chapter-level accuracy |
| E-004 | Add dialogue as supplementary input | Tests whether patient transcript adds ICD-10 signal |
| E-005 | Markdown stripping before tokenisation | Frees context window of `**` tokens |
| E-006 | Weighted cross-entropy | Forces model to prioritise rare codes |

> **STATUS: E-002 COMPLETE.** Official results — Macro F1 = 0.371,
> Accuracy = 49.6%, Top-5 = 77.5%. Coverage-adjusted Macro F1 ≈ 0.74,
> comparable to E-001's 0.760. Best checkpoint: epoch 19.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔍 Phase 11: Confusion Matrix & Prediction Analysis (E-002)

Evaluates the model on the held-out test set and analyses prediction
distribution and confusion patterns. Two key questions:

1. **Has the model collapsed?** A mode-collapsed model predicts the same
   code for every record. Diversity score confirms whether the model is
   making genuinely distributed predictions across the 1,926-class space.

2. **Where do errors concentrate?** The top 10 most confused codes reveal
   whether errors are random or structured within clinical chapters —
   the same analysis performed for E-001.

Note that with ~1 record per class in the test set, the confusion matrix
is extremely sparse. Most cells will be 0 or 1. The top 10 analysis
focuses on the codes that accumulate the most misclassifications in
absolute terms.

</div>

In [ ]:
# ==============================================================================
# PHASE 11: CONFUSION MATRIX & PREDICTION ANALYSIS (E-002)
# ==============================================================================

import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------------------------
# 1. PREDICT ON TEST SET
# ------------------------------------------------------------------------------
print(f"🔍 Running predictions on test set ({len(tokenized_datasets['test']):,} records)...")
predictions  = trainer.predict(tokenized_datasets["test"])
y_true       = predictions.label_ids
y_pred       = np.argmax(predictions.predictions, axis=-1)

print(f"   ✅ Predictions complete")
print(f"   📊 Test metrics:")
for k, v in predictions.metrics.items():
    print(f"      {k}: {v:.4f}")

# ------------------------------------------------------------------------------
# 2. PREDICTION DISTRIBUTION ANALYSIS
# ------------------------------------------------------------------------------
pred_labels = [id2label[int(i)] for i in y_pred]
true_labels = [id2label[int(i)] for i in y_true]

pred_dist = Counter(pred_labels)
true_dist = Counter(true_labels)

unique_preds = len(set(pred_labels))
print(f"\n📊 Prediction diversity: {unique_preds:,} unique codes predicted "
      f"out of {num_labels:,} possible ({unique_preds/num_labels*100:.1f}%)")

if unique_preds == 1:
    print(f"🚨 CRITICAL: Mode collapse — model predicting only "
          f"'{pred_dist.most_common(1)[0][0]}'")
elif unique_preds < num_labels * 0.1:
    print(f"⚠️  WARNING: Low diversity — model predicting fewer than "
          f"10% of possible classes")
else:
    print(f"   ✅ No mode collapse detected")

print(f"\n📊 TOP 10 MOST PREDICTED CODES:")
print(f"   {'Code':12s}  {'Predicted':>10s}  {'Actual':>10s}")
print(f"   {'─'*38}")
for code, count in pred_dist.most_common(10):
    actual_count = true_dist.get(code, 0)
    print(f"   {code:12s}  {count:>10,}  {actual_count:>10,}")

print(f"\n📊 TOP 10 MOST FREQUENT TRUE CODES:")
print(f"   {'Code':12s}  {'Actual':>10s}  {'Predicted':>10s}")
print(f"   {'─'*38}")
for code, count in true_dist.most_common(10):
    pred_count = pred_dist.get(code, 0)
    print(f"   {code:12s}  {count:>10,}  {pred_count:>10,}")

# ------------------------------------------------------------------------------
# 3. CONFUSION MATRIX — TOP 10 MOST CONFUSED
# ------------------------------------------------------------------------------
present_indices = np.unique(np.concatenate([y_true, y_pred]))
present_labels  = [id2label[int(i)] for i in present_indices]

cm    = confusion_matrix(y_true, y_pred, labels=present_indices)
cm_df = pd.DataFrame(cm, index=present_labels, columns=present_labels)

confusion_mass = cm_df.values.copy()
np.fill_diagonal(confusion_mass, 0)
row_sums         = confusion_mass.sum(axis=1)
top_confused_idx = np.argsort(row_sums)[-10:]
top_labels       = [present_labels[i] for i in top_confused_idx]

print(f"\n🚨 Top 10 most confused ICD-10 codes:")
for i, label in enumerate(reversed(top_labels)):
    errors = int(row_sums[present_labels.index(label)])
    print(f"   {i+1:2d}. {label:12s} — {errors:,} misclassified records")

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_df.loc[top_labels, top_labels],
    annot=True, fmt='d', cmap="YlOrRd",
    linewidths=0.5
)
plt.title(
    f"Top 10 Confused ICD-10 Codes\n{cfg['experiment_name']}",
    fontsize=14, fontweight='bold'
)
plt.xlabel("Predicted Code")
plt.ylabel("True Code")
plt.tight_layout()

cm_filename = f"{cfg['experiment_name']}_confusion_matrix.png"
plt.savefig(EXP_DIR / cm_filename, dpi=150, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------------------
# 4. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 11: Confusion Matrix",
    action="confusion_matrix_generated",
    details={
        "test_size":        len(tokenized_datasets["test"]),
        "unique_preds":     unique_preds,
        "total_classes":    num_labels,
        "top_confused":     top_labels,
        "test_macro_f1":    predictions.metrics.get("test_macro_f1"),
        "test_accuracy":    predictions.metrics.get("test_accuracy"),
        "test_top5":        predictions.metrics.get("test_top_5_accuracy"),
    },
    notebook="03-Model_ClinicalBERT_Surgical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 11 complete")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔍 Phase 11b: Chapter-Level Confusion Analysis

The standard ICD-10 code-level confusion matrix is uninformative for E-002
because the test set contains only 1 record per class — every
misclassification produces a single off-diagonal cell and the 10×10 submatrix
of most confused codes shows all zeros.

A chapter-level analysis collapses both true and predicted codes to their
ICD-10 chapter (first letter, A–Z) and asks the clinically meaningful
question: when the model predicts the wrong ICD-10 code, does it stay
within the correct clinical chapter or cross into an unrelated one?

### Why This Matters

A model that confuses `M25.562` (left knee pain) with `M79.3` (panniculitis)
has made a within-chapter error — it understands the patient has a
musculoskeletal condition but picked the wrong specific code. This is
clinically far less harmful than confusing `M25.562` with `F32.9`
(major depressive disorder), which is a cross-chapter error indicating
the model has fundamentally misread the clinical presentation.

The within-chapter vs cross-chapter error breakdown directly informs
whether a hierarchical two-stage architecture would help — if most errors
are within-chapter, a Stage-1 chapter router followed by a Stage-2
within-chapter resolver would address the primary failure mode.

### Outputs

- **Chapter confusion matrix** — 22×22 heatmap showing where predictions
  land at the chapter level
- **Per-chapter accuracy bar chart** — identifies which clinical chapters
  the model handles well vs poorly
- **Error breakdown** — quantifies within-chapter vs cross-chapter errors
  as a percentage of all predictions

</div>

In [ ]:
# ==============================================================================
# PHASE 11b: CHAPTER-LEVEL CONFUSION ANALYSIS (E-002)
# ==============================================================================
# The standard ICD-10 confusion matrix is uninformative at the code level
# because the test set has only 1 record per class — every misclassification
# is a 100% miss for that class and all off-diagonal cells show 0 or 1.
#
# A chapter-level analysis collapses both true and predicted codes to their
# ICD-10 chapter (first letter) and asks: when the model is wrong, does it
# stay within the correct clinical chapter or cross into an unrelated one?
#
# This reveals the same "semantic neighbourhood" pattern we found in E-001
# but at the chapter level rather than the category level.
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# ------------------------------------------------------------------------------
# 1. COLLAPSE TO CHAPTER LEVEL
# ------------------------------------------------------------------------------
# y_true and y_pred are already in scope from Phase 11

pred_labels_list = [id2label[int(i)] for i in y_pred]
true_labels_list  = [id2label[int(i)] for i in y_true]

# Extract chapter (first character of ICD-10 code)
pred_chapters = [code[0] for code in pred_labels_list]
true_chapters  = [code[0] for code in true_labels_list]

all_chapters = sorted(set(true_chapters + pred_chapters))

print(f"📊 Chapter-level analysis:")
print(f"   True chapters present:      {len(set(true_chapters))}")
print(f"   Predicted chapters present: {len(set(pred_chapters))}")
print(f"   Total chapters:             {len(all_chapters)}")

# ------------------------------------------------------------------------------
# 2. BUILD CHAPTER-LEVEL CONFUSION MATRIX
# ------------------------------------------------------------------------------
chapter_to_idx = {ch: i for i, ch in enumerate(all_chapters)}

cm_chapter = np.zeros((len(all_chapters), len(all_chapters)), dtype=int)
for true_ch, pred_ch in zip(true_chapters, pred_chapters):
    cm_chapter[chapter_to_idx[true_ch]][chapter_to_idx[pred_ch]] += 1

cm_chapter_df = pd.DataFrame(
    cm_chapter,
    index=all_chapters,
    columns=all_chapters
)

# ------------------------------------------------------------------------------
# 3. COMPUTE CHAPTER-LEVEL ACCURACY
# ------------------------------------------------------------------------------
chapter_correct = sum(1 for t, p in zip(true_chapters, pred_chapters) if t == p)
chapter_accuracy = chapter_correct / len(true_chapters)

print(f"\n   📊 Chapter-level accuracy: {chapter_accuracy:.1%}")
print(f"   📊 Code-level accuracy:    {sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true):.1%}")
print(f"\n   ℹ️  When the model is wrong at the code level,")
print(f"      it stays in the correct ICD-10 chapter "
      f"{chapter_accuracy - sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true):.1%} "
      f"of the time more than random.")

# ------------------------------------------------------------------------------
# 4. WITHIN-CHAPTER vs CROSS-CHAPTER ERROR ANALYSIS
# ------------------------------------------------------------------------------
within_chapter_errors = 0
cross_chapter_errors  = 0
correct_predictions   = 0

for true_code, pred_code in zip(true_labels_list, pred_labels_list):
    if true_code == pred_code:
        correct_predictions += 1
    elif true_code[0] == pred_code[0]:
        within_chapter_errors += 1
    else:
        cross_chapter_errors += 1

total = len(true_labels_list)
print(f"\n📊 ERROR BREAKDOWN:")
print(f"   ✅ Correct predictions:      {correct_predictions:,} ({correct_predictions/total:.1%})")
print(f"   ℹ️  Within-chapter errors:   {within_chapter_errors:,} ({within_chapter_errors/total:.1%})")
print(f"   ⚠️  Cross-chapter errors:    {cross_chapter_errors:,} ({cross_chapter_errors/total:.1%})")
print(f"\n   ℹ️  Of all errors, {within_chapter_errors/(within_chapter_errors+cross_chapter_errors):.1%} "
      f"stay within the correct clinical chapter")

# ------------------------------------------------------------------------------
# 5. CHAPTER-LEVEL PERFORMANCE TABLE
# ------------------------------------------------------------------------------
print(f"\n📊 PER-CHAPTER PERFORMANCE:")
print(f"   {'Chapter':8s}  {'True':>6s}  {'Correct':>8s}  {'Accuracy':>9s}  {'Top Chapter Confusion'}")
print(f"   {'─'*65}")

chapter_stats = []
for ch in all_chapters:
    true_mask    = [i for i, t in enumerate(true_chapters) if t == ch]
    n_true       = len(true_mask)
    n_correct    = sum(1 for i in true_mask if pred_chapters[i] == ch)
    accuracy     = n_correct / n_true if n_true > 0 else 0

    # Most common wrong chapter
    wrong_preds  = [pred_chapters[i] for i in true_mask if pred_chapters[i] != ch]
    top_wrong    = Counter(wrong_preds).most_common(1)
    top_wrong_str = f"→ {top_wrong[0][0]} ({top_wrong[0][1]}x)" if top_wrong else "—"

    chapter_stats.append({
        "chapter": ch, "n_true": n_true,
        "n_correct": n_correct, "accuracy": accuracy,
        "top_wrong": top_wrong_str
    })
    print(f"   {ch:8s}  {n_true:>6,}  {n_correct:>8,}  {accuracy:>8.1%}  {top_wrong_str}")

# ------------------------------------------------------------------------------
# 6. VISUALISE CHAPTER-LEVEL CONFUSION MATRIX
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Full chapter confusion matrix
sns.heatmap(
    cm_chapter_df,
    annot=True, fmt='d', cmap="YlOrRd",
    linewidths=0.3,
    ax=axes[0]
)
axes[0].set_title(
    f"Chapter-Level Confusion Matrix\n{cfg['experiment_name']}",
    fontsize=13, fontweight='bold'
)
axes[0].set_xlabel("Predicted Chapter")
axes[0].set_ylabel("True Chapter")

# Per-chapter accuracy bar chart
chapter_df = pd.DataFrame(chapter_stats).sort_values("accuracy", ascending=True)
axes[1].barh(chapter_df["chapter"], chapter_df["accuracy"],
             color=['#e74c3c' if a < 0.3 else '#f39c12' if a < 0.6 else '#27ae60'
                    for a in chapter_df["accuracy"]])
axes[1].set_xlabel("Chapter-Level Accuracy")
axes[1].set_title("Per-Chapter Accuracy\n(red < 30%, amber < 60%, green ≥ 60%)",
                  fontsize=13, fontweight='bold')
axes[1].axvline(x=chapter_accuracy, color='white', linestyle='--',
                linewidth=1.5, label=f"Overall: {chapter_accuracy:.1%}")
axes[1].legend()
axes[1].set_xlim(0, 1)

plt.tight_layout()

# Save
cm_chapter_filename = f"{cfg['experiment_name']}_chapter_confusion.png"
plt.savefig(EXP_DIR / cm_chapter_filename, dpi=150, bbox_inches='tight')

registry_path = (config.resolve_path("outputs", "evaluations")
                 / "registry" / cfg['experiment_name'])
if registry_path.exists():
    plt.savefig(registry_path / cm_chapter_filename, dpi=150, bbox_inches='tight')
    print(f"\n   ✅ Chapter confusion matrix saved to registry")

plt.show()

# ------------------------------------------------------------------------------
# 7. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 11b: Chapter Confusion",
    action="chapter_confusion_generated",
    details={
        "chapter_accuracy":       round(chapter_accuracy, 4),
        "within_chapter_errors":  within_chapter_errors,
        "cross_chapter_errors":   cross_chapter_errors,
        "within_chapter_error_pct": round(
            within_chapter_errors /
            (within_chapter_errors + cross_chapter_errors), 4
        ) if (within_chapter_errors + cross_chapter_errors) > 0 else 0,
    },
    notebook="03-Model_ClinicalBERT_Surgical_ICD10"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 11b complete")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔍 E-002 Chapter-Level Confusion Interpretation

### The Central Finding

The gap between chapter-level accuracy (82.9%) and code-level accuracy
(46.9%) is the most important result from E-002. It reveals that the model
has learned the clinical domain structure of ICD-10 — it knows which
chapter a patient belongs to 83% of the time — but struggles to resolve
the specific code within that chapter.

| Level | Accuracy | Interpretation |
|---|---|---|
| Chapter (A–Z) | 82.9% | Model understands clinical domain |
| Code (full ICD-10) | 46.9% | Model struggles with within-chapter resolution |
| Gap | 36.0pp | Within-chapter discrimination is the limiting factor |

### Error Structure

Of all 513 incorrect predictions:

- **67.8% stay within the correct clinical chapter** — the model picked
  the wrong specific code but the right clinical domain
- **32.2% cross chapter boundaries** — the model fundamentally
  misidentified the clinical domain

This structured error pattern is the strongest possible evidence for a
hierarchical two-stage architecture. A Stage-1 chapter router achieving
83% accuracy followed by a Stage-2 within-chapter resolver would address
the primary failure mode directly.

### Chapter Performance Analysis

**Outstanding (>90% chapter accuracy):**
F-codes (Mental disorders, 95.9%), L-codes (Skin, 97.0%), N-codes
(Genitourinary, 96.7%), H-codes (Eye/Ear, 94.9%), J-codes (Respiratory,
94.7%), M-codes (Musculoskeletal, 94.2%). These chapters have distinctive
clinical language that Bio_ClinicalBERT resolves reliably even at the
specific code level.

**Problematic chapters:**

- **Z-codes (48.9%, 137 records)** — the largest single chapter
  (263 classes, 13.7% of label space) and the worst performing. Z-codes
  cover administrative and contextual health encounters — routine
  examinations, screenings, follow-up care — with highly similar clinical
  language across subclasses. The dominant confusion `Z → N (12x)` likely
  reflects obstetric and gynaecological encounters coded as health contact
  (Z) being predicted as genitourinary conditions (N). Z-codes represent
  the hardest within-chapter discrimination problem in the dataset.

- **R-codes (63.2%, 87 records)** — symptom and sign codes are inherently
  ambiguous. By definition they describe clinical presentations without
  specifying a definitive aetiology, so they naturally share language with
  the definitive diagnosis codes they often precede. The `R → M (6x)`
  confusion reflects pain and symptom codes being resolved to
  musculoskeletal diagnoses — clinically plausible but technically wrong.

- **O-codes (75.0%, 36 records)** — obstetric codes, with `O → Z (5x)`
  confusion reflecting obstetric encounters being predicted as routine
  health contact visits. The clinical language for obstetric monitoring
  closely resembles Z-coded routine antenatal care.

### What This Means for Architecture

The chapter-level analysis provides a clear roadmap for E-003:

**Stage 1 (Chapter Router):** Train a 22-way chapter classifier. With
~44 training examples per chapter on average and distinctive
chapter-level language, this should achieve >90% accuracy — well above
the current 82.9% flat chapter accuracy.

**Stage 2 (Within-Chapter Resolver):** Train separate within-chapter
classifiers using the Stage-1 predictions as routing signals. Each
resolver sees only the codes from one chapter, dramatically reducing
the effective label space (e.g. 222 M-codes → one resolver, 263 Z-codes
→ one resolver).

**Priority targets for Stage-2:** Z-codes and R-codes have the largest
within-chapter discrimination gap and would benefit most from focused
resolvers with access to more specific training signal.

> **E-002 Conclusion:** Bio_ClinicalBERT has learned the ICD-10 chapter
> structure with 82.9% accuracy using only 4 training examples per code.
> The primary failure mode is within-chapter code resolution, not
> cross-chapter misidentification. This directly motivates the
> hierarchical two-stage architecture for E-003.

</div>